In [13]:
import zipfile
import tempfile
import shutil

import pandas as pd
import numpy as np
from io import BytesIO
from collections import defaultdict
import matplotlib.pyplot as plt
import os
import re
import time

In [ ]:
df_og = pd.read_csv(r"C:\Users\legen\Desktop\Lab Project\BC\data\market_df_merged.csv", encoding='utf-8-sig')
df_og = df_og.drop(columns = 'Unnamed: 0')

In [ ]:
df_og.head()

,행정기관코드,시장명,시도,시군구,위도,경도,시장면적,전체점포,노점수,점포상인,...,subway,tour,item_diversity,is_food_based,market_item_type,지원금액,지원여부,has_assoc,delivery_grocery,빈점포율
0,3114051000,(주)신정시장,울산광역시,남구,35.542374,129.310261,1668,90,4,86,...,0,0,7.0,1.0,2,82.5,1.0,1.0,0,0.044444
1,3114062500,수암상가시장,울산광역시,남구,35.526896,129.320722,6628,83,49,83,...,0,0,8.0,1.0,2,0.0,0.0,1.0,0,0.000000
2,3114062500,수암종합시장,울산광역시,남구,35.527650,129.320771,2205,52,43,46,...,0,0,5.0,1.0,2,0.0,0.0,1.0,0,0.038462
3,3114051000,신정상가시장,울산광역시,남구,35.541932,129.309097,22230,252,100,249,...,0,0,9.0,1.0,2,0.0,0.0,1.0,1,0.011905
4,3114051000,신정평화시장,울산광역시,남구,35.538326,129.303564,1198,62,0,46,...,0,0,6.0,1.0,2,153.0,1.0,1.0,0,0.258065


In [7]:
HOLIDAY_DATE = set([
    20230902, 20230903, 20230909, 20230910, 20230916, 20230917, 20230923,
    20230924, 20230928, 20230929, 20230930, 20231001, 20231002, 20231003,
    20231007, 20231008, 20231009, 20231014, 20231015
])

In [8]:
AGE_MAP = {
    0: 0,
    1: 10,
    2: 20,
    3: 30,
    4: 40,
    5: 50,
    6: 60,
    7: 70,
    8: 80
}

def age_bucket(p):
    try:
        p_int = int(p)
    except Exception:
        return "age_unknown"
    return AGE_MAP.get(p_int, "age_unknown")



# purpose 그룹
PURPOSE_MAP = {
    0: "귀가",
    1: "업무",
    2: "학업",
    3: "쇼핑여가",
    4: "기타",
    5: "여행",
}

def purpose_bucket(p):
    try:
        p_int = int(p)
    except Exception:
        return "purpose_unknown"
    return PURPOSE_MAP.get(p_int, "purpose_unknown")

# holiday 그룹
def day_bucket(d):
    return "holiday" if d in HOLIDAY_DATE else "weekday"


In [9]:
def iter_inner_csv_members_tempfile(outer_zip_path, inner_zip_name, csv_prefix=None):
    # od_20230901_10.zip 등을 임시파일로 저장한 뒤 내부 CSV member 이름을 yield
    with zipfile.ZipFile(outer_zip_path, "r") as outer:
        with outer.open(inner_zip_name) as src:
            fd, tmp_zip_path = tempfile.mkstemp(suffix=".zip")
            os.close(fd)
            try:
                with open(tmp_zip_path, "wb") as f:
                    shutil.copyfileobj(src, f, length=1024 * 1024)  # 1MB 단위 복사

                with zipfile.ZipFile(tmp_zip_path, "r") as inner:
                    members = [m for m in inner.namelist() if m.endswith(".csv")]
                    if csv_prefix is not None:
                        members = [m for m in members if os.path.basename(m).startswith(csv_prefix)]
                    for m in sorted(members):
                        yield tmp_zip_path, m  # tmp_zip_path는 나중에 inner를 다시 열기 위해 반환
            finally:
                if os.path.exists(tmp_zip_path):
                    os.remove(tmp_zip_path)

In [10]:
def export_od_population_by_code(
    outer_zip_path,
    code,
    inner_zip_names,
    out_dir,
    usecols=None,
    chunksize=500_000
):
    # 1개의 행정동씩 dest_hdong_cd==code만 필터링해서 저장(append)
    code = str(code)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"od_population_{code}.csv")

    if usecols is None:
        usecols = ["origin_hdong_cd","dest_hdong_cd","date","age","dest_purpose","modal","od_cnts"]

    first_write = True

    for inner_zip_name in inner_zip_names:
        # inner zip 안의 csv members 순회
        for tmp_zip_path, csv_member in iter_inner_csv_members_tempfile(
            outer_zip_path, inner_zip_name, csv_prefix="od_"
        ):
            with zipfile.ZipFile(tmp_zip_path, "r") as inner:
                with inner.open(csv_member) as f:
                    for chunk in pd.read_csv(f, usecols=usecols, chunksize=chunksize):
                        chunk["dest_hdong_cd"] = chunk["dest_hdong_cd"].astype(str)
                        chunk = chunk[chunk["dest_hdong_cd"] == code]
                        if chunk.empty:
                            continue

                        chunk.to_csv(out_path, index=False, mode="w" if first_write else "a",
                                     header=first_write, encoding="utf-8-sig")
                        first_write = False

    return out_path


In [11]:
outer_zip_path = r"F:\현일\데이터 및 문제\데이터분석 분야_로우데이터.zip"

# market_df 준비
market_df = df_og.copy()
market_df["행정기관코드"] = market_df["행정기관코드"].astype(str)
code_list = market_df["행정기관코드"].astype(str).unique().tolist()

OD_INNER_ZIPS = [
    "od_20230901_10.zip",
    "od_20230911_20.zip",
    "od_20230921_30.zip",
    "od_20231001_15.zip",
]
STAY_INNER_ZIPS = [
    "stay_20230901_15.zip",
    "stay_20230916_30.zip",
    "stay_20231001_15.zip",
]

## OD 집계

In [12]:
def extract_od_features(
    outer_zip_path,
    target_codes,
    inner_zip_names,
    chunksize=500_000
):
    
    target_set = set(map(str, target_codes))

    usecols = ["dest_hdong_cd","date","age","dest_purpose","modal","od_cnts"]

    # acc[code][feature] = value 누적
    acc = defaultdict(lambda: defaultdict(float))

    for inner_zip_name in inner_zip_names:
        for tmp_zip_path, csv_member in iter_inner_csv_members_tempfile(
            outer_zip_path, inner_zip_name, csv_prefix="od_"
        ):
            with zipfile.ZipFile(tmp_zip_path, "r") as inner:
                with inner.open(csv_member) as f:
                    for df in pd.read_csv(f, usecols=usecols, chunksize=chunksize):
                        df["dest_hdong_cd"] = df["dest_hdong_cd"].astype(str)
                        df = df[df["dest_hdong_cd"].isin(target_set)]
                        if df.empty:
                            continue

                        # 날짜 int로 (안되면 string->int 변환)
                        if df["date"].dtype != np.int64 and df["date"].dtype != np.int32:
                            df["date"] = df["date"].astype(int, errors="ignore")

                        # total
                        g_total = df.groupby("dest_hdong_cd")["od_cnts"].sum()
                        for code, v in g_total.items():
                            acc[code]["od_in_total"] += float(v)

                        # purpose 그룹(일상/여가)
                        df["_purpose_grp"] = df["dest_purpose"].apply(purpose_bucket)
                        g_pgrp = df.groupby(["dest_hdong_cd","_purpose_grp"])["od_cnts"].sum()
                        for (code, grp), v in g_pgrp.items():
                            acc[code][f"od_in_{grp}"] += float(v)

                        # age 그룹(4개)
                        df["_age_grp"] = df["age"].apply(age_bucket)
                        g_agrp = df.groupby(["dest_hdong_cd","_age_grp"])["od_cnts"].sum()
                        for (code, grp), v in g_agrp.items():
                            acc[code][f"od_in_{grp}"] += float(v)

                        # modal별(원본 값 유지)
                        g_modal = df.groupby(["dest_hdong_cd","modal"])["od_cnts"].sum()
                        for (code, modal), v in g_modal.items():
                            acc[code][f"od_in_modal_{modal}"] += float(v)

                        # 주중/휴일
                        df["_day_grp"] = df["date"].apply(day_bucket)
                        g_day = df.groupby(["dest_hdong_cd","_day_grp"])["od_cnts"].sum()
                        for (code, grp), v in g_day.items():
                            acc[code][f"od_in_{grp}"] += float(v)

    # dict -> df
    rows = []
    for code in target_set:
        row = {"행정기관코드": code}
        row.update(acc[code])
        rows.append(row)

    return pd.DataFrame(rows).fillna(0)


In [13]:
od_feat = extract_od_features(outer_zip_path, code_list, OD_INNER_ZIPS, chunksize=300_000)

In [ ]:
#od_feat_prev = od_feat.copy()

In [14]:
od_feat

,행정기관코드,od_in_total,od_in_귀가,od_in_기타,od_in_쇼핑여가,od_in_학업,od_in_0,od_in_10,od_in_20,od_in_30,...,od_in_modal_4.0,od_in_modal_3.0,od_in_modal_4,od_in_modal_3,od_in_modal_7.0,od_in_modal_2.0,od_in_modal_7,od_in_modal_2,od_in_modal_5.0,od_in_modal_5
0,4887025000,68927.0,26877.0,23802.0,1773.0,1035.0,24306.0,4153.0,3171.0,7146.0,...,181.0,124.0,24.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,4873025300,481765.0,274300.0,133911.0,955.0,1116.0,68005.0,20359.0,61509.0,247457.0,...,13016.0,3103.0,969.0,323.0,173.0,61.0,25.0,19.0,0.0,0.0
2,4717051000,715372.0,152568.0,150202.0,193259.0,38254.0,47956.0,240819.0,311024.0,33437.0,...,37424.0,9429.0,2735.0,688.0,17.0,23.0,6.0,0.0,5.0,0.0
3,5013032000,134307.0,6292.0,19404.0,373.0,341.0,31862.0,24291.0,11718.0,32996.0,...,146.0,114.0,43.0,12.0,367.0,0.0,26.0,0.0,0.0,0.0
4,4677040000,31163.0,6883.0,14448.0,85.0,0.0,1597.0,438.0,11485.0,12921.0,...,267.0,121.0,26.0,14.0,10.0,0.0,0.0,0.0,5.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
655,4812754500,122000.0,101447.0,13368.0,0.0,1056.0,47200.0,7853.0,14823.0,11926.0,...,10950.0,1478.0,774.0,115.0,0.0,0.0,0.0,0.0,0.0,0.0
656,4122052000,1120888.0,679865.0,113563.0,181776.0,68446.0,169585.0,334964.0,189393.0,215682.0,...,147478.0,24919.0,10540.0,1535.0,63.0,1572.0,0.0,75.0,9258.0,473.0
657,4884025000,105108.0,37621.0,31277.0,6676.0,2129.0,16895.0,9204.0,38702.0,8047.0,...,616.0,247.0,10.0,12.0,25.0,0.0,0.0,0.0,0.0,0.0
658,4575035500,68916.0,11459.0,23190.0,5958.0,469.0,16630.0,8906.0,22180.0,16076.0,...,565.0,209.0,110.0,27.0,11.0,0.0,0.0,0.0,65.0,5.0


## STAY 집계

In [15]:
def extract_stay_features(
    outer_zip_path,
    target_codes,
    inner_zip_names,
    chunksize=500_000
):
    target_set = set(map(str, target_codes))

    usecols = ["hdong_cd","date","time","age","purpose","stay_cnts"]

    acc = defaultdict(lambda: defaultdict(float))

    for inner_zip_name in inner_zip_names:
        for tmp_zip_path, csv_member in iter_inner_csv_members_tempfile(
            outer_zip_path, inner_zip_name, csv_prefix="stay_"
        ):
            with zipfile.ZipFile(tmp_zip_path, "r") as inner:
                with inner.open(csv_member) as f:
                    for df in pd.read_csv(f, usecols=usecols, chunksize=chunksize):
                        df["hdong_cd"] = df["hdong_cd"].astype(str)
                        df = df[df["hdong_cd"].isin(target_set)]
                        if df.empty:
                            continue

                        if df["date"].dtype != np.int64 and df["date"].dtype != np.int32:
                            df["date"] = df["date"].astype(int, errors="ignore")

                        # total
                        g_total = df.groupby("hdong_cd")["stay_cnts"].sum()
                        for code, v in g_total.items():
                            acc[code]["stay_total"] += float(v)

                        # purpose 그룹
                        df["_purpose_grp"] = df["purpose"].apply(purpose_bucket)
                        g_pgrp = df.groupby(["hdong_cd","_purpose_grp"])["stay_cnts"].sum()
                        for (code, grp), v in g_pgrp.items():
                            acc[code][f"stay_{grp}"] += float(v)

                        # age 그룹
                        df["_age_grp"] = df["age"].apply(age_bucket)
                        g_agrp = df.groupby(["hdong_cd","_age_grp"])["stay_cnts"].sum()
                        for (code, grp), v in g_agrp.items():
                            acc[code][f"stay_{grp}"] += float(v)

                        # 시간대별 체류(원본 time 유지)
                        g_time = df.groupby(["hdong_cd","time"])["stay_cnts"].sum()
                        for (code, t), v in g_time.items():
                            acc[code][f"stay_time_{t}"] += float(v)

                        # 주중/휴일
                        df["_day_grp"] = df["date"].apply(day_bucket)
                        g_day = df.groupby(["hdong_cd","_day_grp"])["stay_cnts"].sum()
                        for (code, grp), v in g_day.items():
                            acc[code][f"stay_{grp}"] += float(v)

    rows = []
    for code in target_set:
        row = {"행정기관코드": code}
        row.update(acc[code])
        rows.append(row)

    return pd.DataFrame(rows).fillna(0)


In [16]:
# stay_feat_prev = stay_feat.copy()

In [17]:
stay_feat = extract_stay_features(outer_zip_path, code_list, STAY_INNER_ZIPS, chunksize=300_000)

In [18]:
stay_feat.head(5)

,행정기관코드,stay_total,stay_귀가,stay_기타,stay_업무,stay_학업,stay_0,stay_10,stay_20,stay_30,...,stay_time_18:00,stay_time_19:00,stay_time_20:00,stay_time_21:00,stay_time_22:00,stay_time_23:00,stay_weekday,stay_쇼핑여가,stay_여행,stay_holiday
0,4887025000,12756120.0,8835162.0,2002802.0,664148.0,120529.0,1181074.0,1626393.0,963541.0,1445953.0,...,817558.0,799058.0,784280.0,768588.0,750489.0,734656.0,7302392.0,95636.0,1037843.0,5453728.0
1,4873025300,15251945.0,10252183.0,2398479.0,2044844.0,44547.0,2325708.0,1420282.0,1277258.0,3096324.0,...,940684.0,919807.0,906772.0,893261.0,893443.0,887008.0,9260140.0,43071.0,468821.0,5991805.0
2,4717051000,8252302.0,3250723.0,1411394.0,1027673.0,263918.0,269403.0,1426667.0,1911475.0,829309.0,...,577436.0,549350.0,504778.0,442271.0,384354.0,341622.0,4432407.0,1211812.0,1086782.0,3819895.0
3,5013032000,9102025.0,4507346.0,1751600.0,696705.0,71766.0,595055.0,811663.0,1025643.0,1370207.0,...,539692.0,524540.0,514849.0,504776.0,493200.0,482211.0,5169273.0,39308.0,2035300.0,3932752.0
4,4677040000,2063157.0,1104160.0,561151.0,164661.0,279.0,42021.0,93092.0,174746.0,202208.0,...,119599.0,108040.0,103464.0,101338.0,99815.0,98483.0,1132627.0,476.0,232430.0,930530.0


In [19]:
feat = od_feat.merge(stay_feat, on="행정기관코드", how="outer").fillna(0)

In [20]:
feat["od_in_modal_0"] = (
    feat.get("od_in_modal_0", 0)
    + feat.get("od_in_modal_0.0", 0)
)
feat.drop(columns=["od_in_modal_0.0"], inplace=True, errors="ignore")

feat["od_in_modal_1"] = (
    feat.get("od_in_modal_1", 0)
    + feat.get("od_in_modal_1.0", 0)
)
feat.drop(columns=["od_in_modal_1.0"], inplace=True, errors="ignore")

feat["od_in_modal_2"] = (
    feat.get("od_in_modal_2", 0)
    + feat.get("od_in_modal_2.0", 0)
)
feat.drop(columns=["od_in_modal_2.0"], inplace=True, errors="ignore")

feat["od_in_modal_3"] = (
    feat.get("od_in_modal_3", 0)
    + feat.get("od_in_modal_3.0", 0)
)
feat.drop(columns=["od_in_modal_3.0"], inplace=True, errors="ignore")

feat["od_in_modal_4"] = (
    feat.get("od_in_modal_4", 0)
    + feat.get("od_in_modal_4.0", 0)
)
feat.drop(columns=["od_in_modal_4.0"], inplace=True, errors="ignore")

feat["od_in_modal_5"] = (
    feat.get("od_in_modal_5", 0)
    + feat.get("od_in_modal_5.0", 0)
)
feat.drop(columns=["od_in_modal_5.0"], inplace=True, errors="ignore")

feat["od_in_modal_7"] = (
    feat.get("od_in_modal_7", 0)
    + feat.get("od_in_modal_7.0", 0)
)
feat.drop(columns=["od_in_modal_7.0"], inplace=True, errors="ignore")



In [21]:
feat.columns

Index(['행정기관코드', 'od_in_total', 'od_in_귀가', 'od_in_기타', 'od_in_쇼핑여가',
       'od_in_학업', 'od_in_0', 'od_in_10', 'od_in_20', 'od_in_30', 'od_in_40',
       'od_in_70', 'od_in_modal_0', 'od_in_weekday', 'od_in_업무', 'od_in_50',
       'od_in_60', 'od_in_여행', 'od_in_80', 'od_in_modal_1', 'od_in_holiday',
       'od_in_modal_4', 'od_in_modal_3', 'od_in_modal_7', 'od_in_modal_2',
       'od_in_modal_5', 'stay_total', 'stay_귀가', 'stay_기타', 'stay_업무',
       'stay_학업', 'stay_0', 'stay_10', 'stay_20', 'stay_30', 'stay_40',
       'stay_50', 'stay_60', 'stay_70', 'stay_80', 'stay_time_08:00',
       'stay_time_09:00', 'stay_time_10:00', 'stay_time_11:00',
       'stay_time_12:00', 'stay_time_13:00', 'stay_time_14:00',
       'stay_time_15:00', 'stay_time_16:00', 'stay_time_17:00',
       'stay_time_18:00', 'stay_time_19:00', 'stay_time_20:00',
       'stay_time_21:00', 'stay_time_22:00', 'stay_time_23:00', 'stay_weekday',
       'stay_쇼핑여가', 'stay_여행', 'stay_holiday'],
      dtype='object')

In [22]:
sorted_cols = ['행정기관코드', 
               
               'od_in_total', 'od_in_귀가', 'od_in_기타', 'od_in_쇼핑여가',
               'od_in_업무', 'od_in_여행', 'od_in_학업', 

               #'od_in_age_adole', 'od_in_age_middle', 'od_in_age_senior', 'od_in_age_young',
               'od_in_0', 'od_in_10', 'od_in_20', 'od_in_30', 
               'od_in_40', 'od_in_50', 'od_in_60', 'od_in_70', 
               #'od_in_age_80',

               'od_in_modal_0', 'od_in_modal_1', 'od_in_modal_2', 'od_in_modal_3',
               'od_in_modal_4', 'od_in_modal_5', 'od_in_modal_7', 

               'od_in_weekday', 'od_in_holiday',
              
               'stay_total', 'stay_귀가', 'stay_기타', 'stay_쇼핑여가',
               'stay_업무', 'stay_여행', 'stay_학업', 
              
               #'stay_age_adole', 'stay_age_middle', 'stay_age_senior', 'stay_age_young', 
               'stay_0', 'stay_10', 'stay_20', 'stay_30', 'stay_40',
               'stay_50', 'stay_60', 'stay_70', 'stay_80',
              
               'stay_time_08:00', 'stay_time_09:00', 'stay_time_10:00', 
               'stay_time_11:00', 'stay_time_12:00', 'stay_time_13:00', 
               'stay_time_14:00', 'stay_time_15:00', 'stay_time_16:00', 
               'stay_time_17:00', 'stay_time_18:00', 'stay_time_19:00', 
               'stay_time_20:00', 'stay_time_21:00', 'stay_time_22:00', 
               'stay_time_23:00', 
              
               'stay_weekday','stay_holiday']

eng_cols = [  '행정기관코드', 
               
              'od_in_total', 'od_in_home', 'od_in_other', 'od_in_shop_leisure',
              'od_in_work', 'od_in_travel', 'od_in_edu', 

              #'od_in_age_adole', 'od_in_age_middle', 'od_in_age_senior', 'od_in_age_young',
              'od_in_age_0', 'od_in_age_10', 'od_in_age_20', 'od_in_age_30', 
              'od_in_age_40', 'od_in_age_50', 'od_in_age_60', 'od_in_age_70', 
              #'od_in_age_80',

              'od_in_modal_0', 'od_in_modal_1', 'od_in_modal_2', 'od_in_modal_3',
              'od_in_modal_4', 'od_in_modal_5', 'od_in_modal_7', 

              'od_in_weekday', 'od_in_holiday',

              'stay_total', 'stay_home', 'stay_other', 'stay_shop_leisure',
              'stay_work', 'stay_travel', 'stay_edu', 

              #'stay_age_adole', 'stay_age_middle', 'stay_age_senior', 'stay_age_young', 
              'stay_age_0', 'stay_age_10', 'stay_age_20', 'stay_age_30', 'stay_age_40',
              'stay_age_50', 'stay_age_60', 'stay_age_70', 'stay_age_80',

              'stay_time_08:00', 'stay_time_09:00', 'stay_time_10:00', 
              'stay_time_11:00', 'stay_time_12:00', 'stay_time_13:00', 
              'stay_time_14:00', 'stay_time_15:00', 'stay_time_16:00', 
              'stay_time_17:00', 'stay_time_18:00', 'stay_time_19:00', 
              'stay_time_20:00', 'stay_time_21:00', 'stay_time_22:00', 
              'stay_time_23:00', 

              'stay_weekday','stay_holiday']

In [23]:
feat_eng = feat[sorted_cols].copy()
feat_eng.columns = eng_cols
od_stay_final = feat_eng[eng_cols].copy()

In [25]:
od_stay_final

,행정기관코드,od_in_total,od_in_home,od_in_other,od_in_shop_leisure,od_in_work,od_in_travel,od_in_edu,od_in_age_0,od_in_age_10,...,stay_time_16:00,stay_time_17:00,stay_time_18:00,stay_time_19:00,stay_time_20:00,stay_time_21:00,stay_time_22:00,stay_time_23:00,stay_weekday,stay_holiday
0,1111051500,1328777.0,283247.0,172022.0,486668.0,163939.0,184934.0,37967.0,216718.0,254150.0,...,1370085.0,1197737.0,1061963.0,963842.0,913367.0,841345.0,793326.0,773603.0,10778200.0,7913685.0
1,1111061500,1720325.0,64377.0,156089.0,676367.0,568402.0,225419.0,29671.0,473869.0,454506.0,...,2827360.0,2645060.0,2326152.0,1786926.0,1455832.0,1148540.0,868227.0,674543.0,24237974.0,9815577.0
2,1111063000,540563.0,86940.0,32091.0,141684.0,216225.0,48962.0,14661.0,57784.0,171282.0,...,956589.0,861040.0,718716.0,576691.0,499706.0,445535.0,398457.0,360625.0,9052219.0,3520331.0
3,1114054000,1115554.0,47122.0,67530.0,220324.0,714183.0,50945.0,15450.0,123426.0,54779.0,...,2066660.0,1848865.0,1503342.0,1015498.0,769341.0,565464.0,448141.0,375411.0,19507019.0,5189658.0
4,1114059000,540722.0,35769.0,30710.0,159876.0,259561.0,49195.0,5611.0,191296.0,17352.0,...,1141143.0,1053609.0,931378.0,787577.0,710100.0,618449.0,527092.0,448001.0,9990069.0,4590737.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
655,5181032000,84813.0,21735.0,29825.0,3434.0,4205.0,25420.0,194.0,33081.0,6228.0,...,438150.0,428250.0,419877.0,406073.0,394849.0,382052.0,375777.0,369452.0,3486795.0,3420916.0
656,5181033000,16959.0,4872.0,3990.0,94.0,746.0,7088.0,169.0,9681.0,810.0,...,224358.0,226393.0,225574.0,217921.0,215458.0,211916.0,209139.0,206764.0,1949416.0,1715079.0
657,5182025000,49279.0,17593.0,13651.0,2013.0,5100.0,9299.0,1623.0,15432.0,4579.0,...,311078.0,313665.0,307876.0,293284.0,287953.0,279146.0,272259.0,267032.0,2848736.0,2069487.0
658,5182025300,108130.0,16431.0,33887.0,2686.0,5142.0,49582.0,402.0,13275.0,10604.0,...,317373.0,316004.0,310917.0,299340.0,291403.0,286556.0,282705.0,279603.0,2514380.0,2437939.0


### 중간저장

In [39]:
od_stay_final.to_csv(r"C:\Users\legen\Desktop\Lab Project\BC\data\od_stay_final.csv", encoding='utf-8-sig', index_label= False)

In [40]:
od_stay_final.columns

Index(['행정기관코드', 'od_in_total', 'od_in_home', 'od_in_other',
       'od_in_shop_leisure', 'od_in_work', 'od_in_travel', 'od_in_edu',
       'od_in_age_0', 'od_in_age_10', 'od_in_age_20', 'od_in_age_30',
       'od_in_age_40', 'od_in_age_50', 'od_in_age_60', 'od_in_age_70',
       'od_in_modal_0', 'od_in_modal_1', 'od_in_modal_2', 'od_in_modal_3',
       'od_in_modal_4', 'od_in_modal_5', 'od_in_modal_7', 'od_in_weekday',
       'od_in_holiday', 'stay_total', 'stay_home', 'stay_other',
       'stay_shop_leisure', 'stay_work', 'stay_travel', 'stay_edu',
       'stay_age_0', 'stay_age_10', 'stay_age_20', 'stay_age_30',
       'stay_age_40', 'stay_age_50', 'stay_age_60', 'stay_age_70',
       'stay_age_80', 'stay_time_08:00', 'stay_time_09:00', 'stay_time_10:00',
       'stay_time_11:00', 'stay_time_12:00', 'stay_time_13:00',
       'stay_time_14:00', 'stay_time_15:00', 'stay_time_16:00',
       'stay_time_17:00', 'stay_time_18:00', 'stay_time_19:00',
       'stay_time_20:00', 'stay_time_21

## 변수처리

In [63]:
import re
import numpy as np
import pandas as pd

def build_behavior_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1) 시간대
    time_cols = [c for c in df.columns if c.startswith("stay_time_")]

    def _time_to_hour(col):
        # stay_time_08:00, stay_time_8:00, stay_time_08:00:00 등 방어
        m = re.search(r"stay_time_(\d{1,2})", col)
        return int(m.group(1)) if m else None

    hour_map = {c: _time_to_hour(c) for c in time_cols}
    # 구간 정의: morning 8-11, lunch 12-15, evening 16-19, night 20-23
    buckets = {
        "stay_morning": [c for c,h in hour_map.items() if h is not None and 8  <= h <= 11],
        "stay_lunch":   [c for c,h in hour_map.items() if h is not None and 12 <= h <= 15],
        "stay_evening": [c for c,h in hour_map.items() if h is not None and 16 <= h <= 19],
        "stay_night":   [c for c,h in hour_map.items() if h is not None and 20 <= h <= 23],
    }
    for new_col, cols in buckets.items():
        if cols:
            df[new_col] = df[cols].sum(axis=1)
        else:
            df[new_col] = 0.0


    # 2) 연령 (OD/Stay)
    # adole: 0,10 / young: 20,30 / middle: 40,50 / senior: 60,70,80
    age_groups = {
        "adole": [0, 10],
        "young": [20, 30],
        "middle": [40, 50],
        "senior": [60, 70, 80],
    }

    def _sum_by_suffix(prefix, suffix_list):
        cols = [f"{prefix}{s}" for s in suffix_list if f"{prefix}{s}" in df.columns]
        return df[cols].sum(axis=1) if cols else 0.0

    # OD
    for gname, suffixes in age_groups.items():
        df[f"od_{gname}"] = _sum_by_suffix("od_in_age_", suffixes)

    # Stay
    for gname, suffixes in age_groups.items():
        df[f"stay_{gname}"] = _sum_by_suffix("stay_age_", suffixes)

    # 3) 이동수단 (OD)
    # 0:차량, 1:시내버스, 2:지하철, 3:도보, 4:기타, 5:철도, 6:시외고속버스, 7:항공기
    # bus = 1+6, sub = 2+5, etc = 4+7
    modal_groups = {
        "car":  [0],
        "walk": [3],
        "bus":  [1, 6],
        "sub":  [2, 5],
        "etc":  [4, 7],
    }

    for gname, suffixes in modal_groups.items():
        df[f"od_modal_{gname}"] = _sum_by_suffix("od_in_modal_", suffixes)


    # 4) 체류 목적
    # stay_consumption = shop_leisure + travel
    # stay_workedu     = work + edu
    # stay_residential = home + other
    purpose_groups = {
        "consumption": ["shop_leisure", "travel"],
        "workedu":     ["work", "edu"],
        "residential": ["home", "other"],
    }

    # OD
    for grp, suffixes in purpose_groups.items():
        cols = [f"od_in_{s}" for s in suffixes if f"od_in_{s}" in df.columns]
        df[f"od_{grp}"] = df[cols].sum(axis=1) if cols else 0.0

    # Stay
    for grp, suffixes in purpose_groups.items():
        cols = [f"stay_{s}" for s in suffixes if f"stay_{s}" in df.columns]
        df[f"stay_{grp}"] = df[cols].sum(axis=1) if cols else 0.0

    return df

# 사용 예시
# df = pd.read_csv("/mnt/data/market_od.csv")
# df = build_behavior_features(df)


In [85]:
od_stay_final_merged = build_behavior_features(od_stay_final)

In [87]:
od_stay_final_merged.to_csv(r"C:\Users\legen\Desktop\Lab Project\BC\data\od_stay_final_merged.csv", encoding='utf-8-sig', index_label=False)

## market 데이터와 병합

In [15]:
import pandas as pd
od_stay_final_merged = pd.read_csv(r"C:\Users\legen\Desktop\Lab Project\BC\data\od_stay_final_merged.csv", encoding='utf-8-sig')
# od_stay_final = od_stay_final.drop(columns = 'Unnamed: 0',)
df_og = pd.read_csv(r"C:\Users\legen\Desktop\Lab Project\BC\data\market_df_merged.csv", encoding='utf-8-sig')
#df_og = df_og.drop(columns = 'Unnamed: 0')


In [16]:
col_lists = [
    # 연령대 유입: 4개
    "od_adole", "od_young", "od_middle", "od_senior",

    # 유입목적: 3개
    'od_consumption', 'od_workedu',	'od_residential',

    # 주중/주말 유입: 2개
    'od_in_weekday', 'od_in_holiday', 
    
    # 유입 시 사용한 이동수단: 5개
    "od_modal_car", "od_modal_walk", "od_modal_bus", "od_modal_sub", "od_modal_etc",
    


    # 연령대 체류: 4개
    "stay_adole", "stay_young", "stay_middle", "stay_senior",

    # 체류목적: 3개
    'stay_consumption', 'stay_workedu',	'stay_residential',

    # 주중/주말 체류: 2개
    'stay_weekday', 'stay_holiday',

    # 시간대 체류: 4개
    'stay_morning', 'stay_lunch', 'stay_evening', 'stay_night'
]

In [17]:
od_stay_final_merged['행정기관코드'] = od_stay_final_merged['행정기관코드'].astype(str)
df_og['행정기관코드'] = df_og['행정기관코드'].astype(str)


market_od = df_og.merge(
    od_stay_final_merged[['행정기관코드'] + col_lists],
    on='행정기관코드',
    how='left'
)

In [18]:
market_od.head(3)

,행정기관코드,시장명,시도,시군구,위도,경도,시장면적,전체점포,노점수,총시장상인,...,stay_senior,stay_consumption,stay_workedu,stay_residential,stay_weekday,stay_holiday,stay_morning,stay_lunch,stay_evening,stay_night
0,3114051000,(주)신정시장,울산광역시,남구,35.542374,129.310261,1668,90,4,110,...,2576689.0,914348.0,939203.0,8430593.0,5992808.0,4291336.0,2578195.0,2470881.0,2600765.0,2634303.0
1,3114062500,수암상가시장,울산광역시,남구,35.526896,129.320722,6628,83,49,191,...,2347591.0,774279.0,766242.0,15200649.0,9830616.0,6910554.0,4072151.0,3833011.0,4244467.0,4591541.0
2,3114062500,수암종합시장,울산광역시,남구,35.527650,129.320771,2205,52,43,103,...,2347591.0,774279.0,766242.0,15200649.0,9830616.0,6910554.0,4072151.0,3833011.0,4244467.0,4591541.0


In [19]:
market_od.columns

Index(['행정기관코드', '시장명', '시도', '시군구', '위도', '경도', '시장면적', '전체점포', '노점수',
       '총시장상인', '편의시설수', '점포_대_상인_비율', 'parking', 'bus', 'mart', 'tour',
       'conv', 'subway', 'pop_adole', 'pop_young', 'pop_middle', 'pop_senior',
       '지원여부', 'has_assoc', 'join_stores', 'item_diversity', 'is_food_based',
       'has_nonfood', 'market_item_type', 'delivery_grocery', '빈점포율',
       'od_adole', 'od_young', 'od_middle', 'od_senior', 'od_consumption',
       'od_workedu', 'od_residential', 'od_in_weekday', 'od_in_holiday',
       'od_modal_car', 'od_modal_walk', 'od_modal_bus', 'od_modal_sub',
       'od_modal_etc', 'stay_adole', 'stay_young', 'stay_middle',
       'stay_senior', 'stay_consumption', 'stay_workedu', 'stay_residential',
       'stay_weekday', 'stay_holiday', 'stay_morning', 'stay_lunch',
       'stay_evening', 'stay_night'],
      dtype='object')

## 저장

In [20]:
market_od.to_csv(r"C:\Users\legen\Desktop\Lab Project\BC\data\market_od.csv", encoding='utf-8-sig', index_label=False)

In [21]:
market_od = pd.read_csv(r"C:\Users\legen\Desktop\Lab Project\BC\data\market_od.csv", encoding='utf-8-sig')

In [ ]:
#market_od_support = market_od[market_od['지원여부'] == 1]
#market_od_all = market_od[market_od['지원여부'] == 0]

In [ ]:
#market_od_support.to_csv(r"C:\Users\legen\Desktop\Lab Project\BC\data\market_od_support.csv", encoding='utf-8-sig', index_label=False)
#market_od_all.to_csv(r"C:\Users\legen\Desktop\Lab Project\BC\data\market_od_all.csv", encoding='utf-8-sig', index_label=False)